# 3 – Catalog-Entity-Plugin mit Component-Tab

Dieses Notebook erstellt das Frontend-Plugin `component-insights`. Es ergänzt auf Entity-Seiten von `Component`-Entities einen neuen Tab **Insights** und zeigt zentrale Entity-Metadaten an.


> **Voraussetzung:** Die Backstage-Installation liegt unter `~/mybackstage`.
>
> Shell-Zellen werden über Python mit `subprocess` ausgeführt. Vor Änderungen wird jeweils eine Sicherung angelegt.

In [ ]:
from pathlib import Path
ROOT = Path.home() / "mybackstage"
assert ROOT.exists(), f"{ROOT} wurde nicht gefunden"
PLUGIN = ROOT / "plugins/component-insights"
print("Backstage:", ROOT)

## Plugin erzeugen

Führe `yarn new` aus und wähle:

- **plugin**
- Plugin ID: **component-insights**

In [ ]:
import subprocess
if not PLUGIN.exists():
    subprocess.run(["yarn", "new"], cwd=ROOT, check=True)
else:
    print("Plugin-Verzeichnis existiert bereits:", PLUGIN)

## Abhängigkeiten installieren

In [ ]:
import subprocess
subprocess.run(
    ["yarn", "--cwd", "plugins/component-insights", "add",
     "@backstage/frontend-plugin-api",
     "@backstage/plugin-catalog-react",
     "@backstage/catalog-model",
     "@backstage/core-components"],
    cwd=ROOT,
    check=True,
)

## EntityContent Extension implementieren

In [ ]:
from pathlib import Path
import shutil, datetime

src = PLUGIN / "src"
(src / "components/ComponentInsights").mkdir(parents=True, exist_ok=True)

files = {
    src / "plugin.tsx": """import { createFrontendPlugin } from '@backstage/frontend-plugin-api';
import { EntityContentBlueprint } from '@backstage/plugin-catalog-react/alpha';

const componentInsightsContent = EntityContentBlueprint.make({
  name: 'component-insights',
  params: {
    path: '/insights',
    title: 'Insights',
    filter: 'kind:component',
    loader: () =>
      import('./components/ComponentInsights').then(m => (
        <m.ComponentInsights />
      )),
  },
});

export const componentInsightsPlugin = createFrontendPlugin({
  pluginId: 'component-insights',
  extensions: [componentInsightsContent],
});
""",
    src / "index.ts": """export { componentInsightsPlugin as default } from './plugin';
""",
    src / "components/ComponentInsights/ComponentInsights.tsx": """import {
  Content,
  InfoCard,
  StructuredMetadataTable,
} from '@backstage/core-components';
import { useEntity } from '@backstage/plugin-catalog-react';

export const ComponentInsights = () => {
  const { entity } = useEntity();

  const metadata = {
    Name: entity.metadata.name,
    Namespace: entity.metadata.namespace ?? 'default',
    Typ: entity.spec?.type ?? 'nicht definiert',
    Lifecycle: entity.spec?.lifecycle ?? 'nicht definiert',
    Owner: entity.spec?.owner ?? 'nicht definiert',
    System: entity.spec?.system ?? 'nicht definiert',
    Tags: entity.metadata.tags?.join(', ') || 'keine',
  };

  return (
    <Content>
      <InfoCard title="Component Insights">
        <StructuredMetadataTable metadata={metadata} />
      </InfoCard>
    </Content>
  );
};
""",
    src / "components/ComponentInsights/index.ts": """export { ComponentInsights } from './ComponentInsights';
""",
}

for path, content in files.items():
    if path.exists():
        backup = path.with_suffix(path.suffix + f".bak-{datetime.datetime.now():%Y%m%d-%H%M%S}")
        shutil.copy2(path, backup)
    path.write_text(content)
    print("geschrieben:", path.relative_to(ROOT))

## Optional: Tab gruppieren

Im neuen Frontend-System kann `EntityContentBlueprint` über `group` einer Tab-Gruppe zugeordnet werden. Für ein direkt sichtbares Haupt-Tab wird hier bewusst keine Gruppe gesetzt.

## TypeScript prüfen

In [ ]:
import subprocess
subprocess.run(["yarn", "tsc"], cwd=ROOT, check=True)

## Start und Kontrolle

```bash
cd ~/mybackstage
yarn start
```

Öffne anschliessend eine Entity mit `kind: Component`. Auf der Entity-Seite sollte der neue Tab **Insights** vorhanden sein. Bei anderen Entity-Kinds wird der Tab durch `filter: 'kind:component'` nicht angezeigt.